In [1]:
import os
from cognite.client import CogniteClient, ClientConfig
from cognite.client.credentials import OAuthClientCredentials

In [2]:
BASE_URL = "https://aw-was-gp-001.cognitedata.com"
PROJECT = "alimentospolarcomercialca"
TOKEN_URL = "https://datamosaix-prod.us.auth0.com/oauth/token"

# 1. OAuth Credentials configured for Auth0
creds = OAuthClientCredentials(
    token_url=TOKEN_URL,
    client_id=os.getenv("CLIENT_ID", "Cs5Yu6WBAPKmlmVwogyRb8a2etU3yCLV"),
    client_secret=os.getenv("CLIENT_SECRET", "UTyU1jLFMwwd6tKdcz8boczIaeB5Us6H7VOwN_zBCG_xI4aPELdWL3zkjL5uonwk"),
    audience="https://cognitedata.com",  # Required global audience for DataMosaix
    scopes=[]
)

In [3]:
# 2. Client Config
cnf = ClientConfig(
    client_name="superenvases-data-mosaix",
    project=PROJECT,
    credentials=creds,
    base_url=BASE_URL
)

In [4]:
# 3. Instantiate and Test Connection
try:
    client = CogniteClient(cnf)
    spaces = client.data_modeling.spaces.list()
    print("Successfully authenticated via Auth0!")
    print(f"Active spaces found in '{PROJECT}': {[s.space for s in spaces]}")
except Exception as e:
    print("Authentication error:")
    print(e)

Successfully authenticated via Auth0!
Active spaces found in 'alimentospolarcomercialca': []


In [5]:
client = CogniteClient(cnf)

In [ ]:
from cognite.client.data_classes.data_modeling import (
    SpaceApply,
    NodeApply,
    EdgeApply,
    NodeOrEdgeData,  # <--- Added missing class
    DirectRelationReference,
    ViewId
)
# 2. Create the Space
CUSTOM_SPACE = "superenvases_plant"
space = SpaceApply(
    space=CUSTOM_SPACE,
    name="Superenvases Plant Space",
    description="Space for Planta Superenvases asset hierarchy and telemetry"
)

print(f"Creating space '{CUSTOM_SPACE}'...")
client.data_modeling.spaces.apply(space)
print("Space ready.")

Creating space 'superenvases_plant'...
Space ready.


In [8]:
# 2. Setup Definitions
cdm_asset_view = ViewId(space="cdf_cdm", external_id="CogniteAsset", version="v1")
edge_type = DirectRelationReference(space="cdf_cdm", external_id="has-parent")

nodes = []
edges = []

def add_asset(asset_id, name, desc, parent_id=None):
    nodes.append(
        NodeApply(
            space=CUSTOM_SPACE,
            external_id=asset_id,
            sources=[{
                "source": cdm_asset_view,
                "properties": {"name": name, "description": desc}
            }]
        )
    )
    if parent_id:
        edges.append(
            EdgeApply(
                space=CUSTOM_SPACE,
                external_id=f"edge_{asset_id}_to_{parent_id}",
                type=edge_type,
                start_node=DirectRelationReference(space=CUSTOM_SPACE, external_id=asset_id),
                end_node=DirectRelationReference(space=CUSTOM_SPACE, external_id=parent_id)
            )
        )

In [9]:
# --- Root Plant ---
add_asset("facility_planta_superenvases", "Planta Superenvases", "Main manufacturing plant")

In [10]:
# --- Production Lines ---
add_asset("line_l1", "Line L1", "Production Line 1", "facility_planta_superenvases")
add_asset("line_l3", "Line L3", "Production Line 3", "facility_planta_superenvases")

In [11]:
# --- Line 1 Machines ---
add_asset("machine_minster_l1", "MINSTER L1", "Minster press on Line 1", "line_l1")
add_asset("machine_printer_l1", "PRINTER L1", "Printer system on Line 1", "line_l1")

for i in [11, 12, 14, 15, 17, 18]:
    add_asset(f"machine_di{i}", f"D&I{i}", f"D&I machine {i} on Line 1", "line_l1")

for i in range(11, 16):
    add_asset(f"machine_ispray{i}", f"ISPRAY {i}", f"Internal spray {i} on Line 1", "line_l1")

In [12]:
# --- Line 3 Machines ---
add_asset("machine_minster_l3", "MINSTER L3", "Minster press on Line 3", "line_l3")
add_asset("machine_printer31", "PRINTER 31", "Printer 31 on Line 3", "line_l3")
add_asset("machine_printer32", "PRINTER 32", "Printer 32 on Line 3", "line_l3")

for i in range(31, 39):
    add_asset(f"machine_standum{i}", f"STANDUM {i}", f"Standum machine {i} on Line 3", "line_l3")

for i in range(31, 39):
    add_asset(f"machine_ispray{i}", f"ISPRAY {i}", f"Internal spray {i} on Line 3", "line_l3")

In [13]:
# 3. Deploy Hierarchy to Data Mosaix
print(f"Deploying {len(nodes)} assets and {len(edges)} hierarchy connections...")
response = client.data_modeling.instances.apply(nodes=nodes, edges=edges)
print(f"Deployment complete! Successfully applied {len(response.nodes)} nodes and {len(response.edges)} edges.")

Deploying 35 assets and 34 hierarchy connections...


AttributeError: 'dict' object has no attribute 'dump'